# Petőfi-GPT – tanítás Google Colabon (GPU)

Ez a notebook a repó kódjával tanítja a modelleket GPU-n. **Semmit nem kell átírni a kódban:** ugyanaz a `train.py` fut, mint a saját gépeden, csak a `--preset colab` kapcsolóval.

**Előtte:** *Runtime → Change runtime type → T4 GPU* (vagy L4, ill. bármilyen elérhető GPU), majd *Save*. A projekt futásai L4-en készültek.

| Lépés | Mit csinál |
|---|---|
| 1 | ellenőrzi a GPU-t |
| 2 | letölti (klónozza) a repót a GitHubról |
| 3 | csatlakoztatja a Google Drive-ot, hogy a checkpointok megmaradjanak |
| 4 | előállítja az adatot (`prepare_data.py`), és ellenőrzi az ujjlenyomatát |
| 5 | gyors próbák: modell-önteszt + 50 lépéses GPU-tanítás |
| 6 | **a projekt 7 tanítása:** dropout-keresés, ablationök, Differential Transformer |
| 7 | mintaszöveg a végleges GPT-ből |
| 8 | a checkpointok letöltése a saját gépedre (az `analysis.ipynb`-hez) |

## 1. GPU ellenőrzése

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
print("torch", torch.__version__, "| CUDA elérhető:", torch.cuda.is_available())
assert torch.cuda.is_available(), "Nincs GPU: Runtime -> Change runtime type -> T4 GPU"

## 2. A repó letöltése (vagy frissítése, ha már letöltötted)

In [ ]:
import os
%cd /content
if not os.path.exists("petofi-gpt"):
    !git clone https://github.com/janosibenedek/petofi-gpt.git
%cd /content/petofi-gpt
!git pull
!git log --oneline -3

## 3. Google Drive: hogy a checkpointok megmaradjanak

A Colab gépe ideiglenes: ha a munkamenet megszakad, minden fájl elveszik róla. A checkpointokat ezért a Google Drive-odra mentjük (`MyDrive/petofi-gpt/checkpoints`). A `train.py` minden javuláskor ment, így egy megszakadt futásból is megmarad az addigi legjobb modell.

Ha nem akarod a Drive-ot használni, állítsd `USE_DRIVE = False`-ra. Ekkor a fájlok a Colab gépén maradnak, és a végén le kell töltened őket (8. lépés).

In [ ]:
USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = "/content/drive/MyDrive/petofi-gpt/checkpoints"
else:
    OUT = "checkpoints"
os.makedirs(OUT, exist_ok=True)
print("checkpointok helye:", OUT)

## 4. Az adat előállítása

Ugyanaz a szkript, mint a gépeden. Az ujjlenyomat (SHA-256) igazolja, hogy pontosan ugyanabból a forrásfájlból dolgozik, így a train/val/test felosztás is ugyanaz.

Ha `False` jönne ki (a Gutenberg időközben módosította a fájlt), töltsd fel a saját gépeden lévő `data/raw/pg41504.txt`-t a Colab bal oldali *Files* paneljén, és futtasd így: `!python prepare_data.py --raw-file pg41504.txt`.

In [ ]:
!python prepare_data.py
import json
meta = json.load(open("data/meta.json", encoding="utf-8"))
EXPECTED = "aa7655a76f0321eaf42d2a56f97b9ddbc7af9274ece654f36d3d3a119c5bffdf"
print("ujjlenyomat egyezik a helyi futással:", meta["raw_sha256"] == EXPECTED)
print({k: v for k, v in meta["splits"].items()})

## 5. Gyors próbák

1. A modell öntesztje (ugyanaz, mint a gépeden).
2. **50 lépéses próbatanítás GPU-n**, a Colab-presettel. Ez ellenőrzi, hogy működik-e a GPU-s ág (16 bites *mixed precision*), és megmutatja, mennyi ideig tart egy lépés. Ebből becsülhető a teljes futás ideje.

In [ ]:
!python model.py
# the smoke-test checkpoint is not needed later: it stays on the temporary Colab disk, not on Drive
!python train.py --preset colab --max-iters 50 --eval-interval 25 --eval-iters 5 --name colab_smoke --out-dir checkpoints

## 6. A projekt tanításai

A Colab-preset: 6 blokk, 384 dimenzió, 6 fej, 256 karakteres kontextus, kb. 10,8 millió paraméter. Tanítás: batch 64, lr 1e-3, legfeljebb 6000 lépés, early stopping a val loss alapján, seed 1337. Minden futás ugyanaz a `train.py`, csak a kapcsolók térnek el:

| # | Cél | Kapcsolók | Checkpoint |
|---|---|---|---|
| 1 | dropout-keresés (a preset alapértéke, 0,2) | – | `colab.pt` |
| 2 | dropout-keresés | `--dropout 0.3` | `colab_drop03.pt` → **végleges GPT** |
| 3 | dropout-keresés | `--dropout 0.4` | `colab_drop04.pt` |
| 4 | ablation: ReLU a GELU helyett | `--dropout 0.3 --activation relu` | `colab_drop03_relu.pt` |
| 5 | ablation: residual kapcsolat nélkül | `--dropout 0.3 --no-residual` | `colab_drop03_nores.pt` |
| 6 | Differential Transformer | `--dropout 0.3 --attention diff` | `colab_drop03_diff.pt` |
| 7 | Differential Transformer | `--dropout 0.2 --attention diff` | `colab_drop02_diff.pt` → **végleges DIFF** |

A végleges modelleket az `analysis.ipynb` választja ki **a val loss alapján**. A DIFF-et azért tanítottuk két dropouttal is, mert 0,3-mal kevésbé tanult túl, mint a GPT, így a GPT-nél jó dropout neki nem feltétlenül megfelelő. Mindkét architektúra így a saját legjobb dropoutjával kerül összevetésre.

A `cpu.pt` (0,8M paraméter) nem itt készül, hanem a saját gépen: `python train.py`.

**Védelem a felülírás ellen:** ha egy futás checkpointja már megvan a Drive-on, a `train` függvény kihagyja, így egy véletlen *Run all* nem írja felül a kész modelleket. Ha egy futás félbeszakadt (megszakadt a munkamenet), a checkpointja egy félkész modell: töröld a Drive-ról, és futtasd újra a cellát.

**Várható idő:** az 5. lépés kiírt idejéből becsülhető (25 lépés ideje × 240 = a 6000 lépés felső becslése). Az early stopping miatt egy futás általában 2600–5000 lépés között leáll.

**Reprodukálhatóság:** a seed rögzített, de a GPU-s számítások (16 bites *mixed precision*) nem bitre pontosan determinisztikusak, ezért egy újratanítás nagyon közeli, de nem pontosan azonos számokat ad.

In [ ]:
def train(name, flags=""):
    # one training run with the colab preset; skipped if its checkpoint already exists,
    # so an accidental 'Run all' never overwrites a finished model
    if os.path.exists(f"{OUT}/{name}.pt"):
        print(f"{name}: a checkpoint már megvan, kihagyva")
        return
    print(f"===== {name} =====")
    !python train.py --preset colab {flags} --name {name} --out-dir "{OUT}"

### 6.1 Dropout-keresés (GPT)

In [ ]:
train("colab")                                   # dropout 0.2, the preset's default
train("colab_drop03", "--dropout 0.3")
train("colab_drop04", "--dropout 0.4")

### 6.2 Ablationök (a kiválasztott 0,3-as dropouttal)

In [ ]:
train("colab_drop03_relu", "--dropout 0.3 --activation relu")
train("colab_drop03_nores", "--dropout 0.3 --no-residual")

### 6.3 Differential Transformer

In [ ]:
train("colab_drop03_diff", "--dropout 0.3 --attention diff")
train("colab_drop02_diff", "--dropout 0.2 --attention diff")

## 7. Mintaszöveg a végleges GPT-ből

In [ ]:
!python generate.py --checkpoint "{OUT}/colab_drop03.pt" --prompt "Szabadság, szerelem!\n" --temperature 0.5 0.8 1.0 --max-new-tokens 400

## 8. A checkpointok letöltése a saját gépedre

Az `analysis.ipynb` mind a 7 checkpointot használja (a `cpu.pt` mellett), ezeknek a gépeden a repó `checkpoints/` mappájában kell lenniük.

**Drive-val:** *Google Drive → petofi-gpt → checkpoints*, jelöld ki a fájlokat, majd *Letöltés* (a Drive egy zip-fájlba csomagolja őket).

**Drive nélkül** ez a cella tölti le őket a böngészővel (fájlonként kb. 40 MB):

In [ ]:
from google.colab import files
for name in ["colab", "colab_drop03", "colab_drop04", "colab_drop03_relu",
             "colab_drop03_nores", "colab_drop03_diff", "colab_drop02_diff"]:
    files.download(f"{OUT}/{name}.pt")